# Conquer Mobile → TripoSR (42 original reference images)
This notebook uses the extracted images in the connected GitHub repository. It does not use paid GPU services. Select **Runtime → Change runtime type → T4 GPU** if a free GPU is available.

In [ ]:
import os, sys, subprocess, pathlib, json, shutil, zipfile
from pathlib import Path
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as e: print('Torch check:', e)
if not ('torch' in sys.modules and torch.cuda.is_available()):
    raise RuntimeError('GPU غير متاح في الجلسة. من Runtime اختَر Change runtime type ثم T4 GPU إذا كان متاحًا مجانًا. لم يتم تشغيل أي خدمة مدفوعة.')

In [ ]:
REPO = Path('/content/Conquer-Mobile-Assets')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets..git',str(REPO)], check=True)
IMAGES = REPO / 'original_images'
assert IMAGES.exists(), 'لم يتم العثور على original_images داخل المستودع'
image_files = sorted([p for p in IMAGES.rglob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg','.webp')])
print('Original images:', len(image_files))
assert len(image_files) == 42, f'المتوقع 42 صورة، الموجود {len(image_files)}'
print('\n'.join(str(p.relative_to(IMAGES)) for p in image_files))

In [ ]:
TSR_REPO = Path('/content/TripoSR')
if not TSR_REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/VAST-AI-Research/TripoSR.git',str(TSR_REPO)], check=True)
os.chdir(TSR_REPO)
# Install project dependencies; this includes the CUDA marching-cubes extension.
subprocess.run([sys.executable,'-m','pip','install','--upgrade','pip','setuptools','wheel'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
print('TripoSR dependencies installed.')

In [ ]:
# Patch official inference script to export an uncolored mesh and save both OBJ and GLB in one model pass.
runpy = TSR_REPO / 'run.py'
src = runpy.read_text()
old = 'meshes = model.extract_mesh(scene_codes, not args.bake_texture, resolution=args.mc_resolution)'
new = 'meshes = model.extract_mesh(scene_codes, False, resolution=args.mc_resolution)'
assert old in src, 'نسخة run.py تغيّرت؛ أوقفت التعديل الآلي للحماية.'
src = src.replace(old, new, 1)
old_export = 'meshes[0].export(out_mesh_path)'
new_export = 'meshes[0].export(out_mesh_path)\n            other_format = "glb" if args.model_save_format == "obj" else "obj"\n            meshes[0].export(os.path.join(output_dir, str(i), f"mesh.{other_format}"))'
assert old_export in src, 'لم أجد سطر التصدير المتوقع.'
src = src.replace(old_export, new_export, 1)
runpy.write_text(src)
print('Patch applied: no vertex colors; OBJ + GLB.')

In [ ]:
# Run images one by one, preserving their original category and filename.
OUT = Path('/content/Conquer_TripoSR_Output')
OUT.mkdir(exist_ok=True)
for idx, image_path in enumerate(image_files, 1):
    rel = image_path.relative_to(IMAGES)
    safe_name = image_path.stem.replace(' ', '_')
    target = OUT / rel.parent / safe_name
    target.mkdir(parents=True, exist_ok=True)
    temp_out = OUT / '_tmp_inference'
    if temp_out.exists(): shutil.rmtree(temp_out)
    cmd = [sys.executable, str(runpy), str(image_path), '--output-dir', str(temp_out), '--model-save-format', 'obj', '--mc-resolution', '256']
    print(f'[{idx}/{len(image_files)}] {rel}')
    subprocess.run(cmd, cwd=str(TSR_REPO), check=True)
    generated = temp_out / '0'
    for ext in ('obj','glb'):
        f = generated / f'mesh.{ext}'
        if not f.exists(): raise FileNotFoundError(f)
        shutil.copy2(f, target / f'{safe_name}.{ext}')
    shutil.rmtree(temp_out)
    del cmd
print('Finished processing all images.')

In [ ]:
# Validate outputs and create one downloadable archive.
models = list(OUT.rglob('*.obj')) + list(OUT.rglob('*.glb'))
obj_count = len(list(OUT.rglob('*.obj')))
glb_count = len(list(OUT.rglob('*.glb')))
print('OBJ:', obj_count, '| GLB:', glb_count)
assert obj_count == 42 and glb_count == 42, 'عدد المخرجات غير مكتمل؛ راجع الأخطاء أعلاه.'
zip_path = Path('/content/Conquer_TripoSR_42_Models.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file(): z.write(f, f.relative_to(OUT))
print('ZIP ready:', zip_path, '| bytes:', zip_path.stat().st_size)
from google.colab import files
files.download(str(zip_path))